In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

: 

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("file:../mlruns")       # Dossier local des runs MLflow
mlflow.set_experiment("fraud_detection_experiment")   # Nom de l'expérience

In [ ]:
data = pd.read_csv("../data/raw/creditcard.csv")
data.head()

In [ ]:
data = pd.read_csv("../data/raw/creditcard.csv")
data.head()

In [ ]:
print("Shape :", data.shape)
print(data.isna().sum().sum(), "valeurs manquantes")
data["Class"].value_counts()

In [ ]:
X = data.drop(columns=["Class"])
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size :", X_train.shape, "Test size :", X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

with mlflow.start_run(run_name="LogReg_Baseline"):
    
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_prob)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("roc_auc", roc_auc)

    mlflow.sklearn.log_model(model, "model")

print("ROC-AUC Logistic Regression:", roc_auc)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="RandomForest_200"):
    
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",
        random_state=42
    )
    rf_model.fit(X_train, y_train)
    
    rf_prob = rf_model.predict_proba(X_test)[:, 1]
    roc_auc_rf = roc_auc_score(y_test, rf_prob)

    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 8)
    mlflow.log_metric("roc_auc", roc_auc_rf)

    mlflow.sklearn.log_model(rf_model, "model")

print("ROC-AUC Random Forest:", roc_auc_rf)

In [ ]:
cm = confusion_matrix(y_test, rf_model.predict(X_test))
print(cm)

plt.figure(figsize=(4,4))
plt.imshow(cm)
plt.title("Confusion Matrix - RandomForest")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.show()

In [ ]:
from subprocess import Popen
import time, webbrowser

print("Launching MLflow UI ...")
mlflow_ui = Popen(["mlflow", "ui"])   # NE PAS TOUCHER !
time.sleep(2)
webbrowser.open("http://127.0.0.1:5000")